# 📦 STEP 1: INSTALL THE TOOLS

## Option A: Install Miniconda (recommended for ML — prebuilt CUDA bins)

# Download & install (silent, no prompts)
wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
bash /tmp/miniconda.sh -b -p $HOME/miniconda3

# Initialize shell (run once)
$HOME/miniconda3/bin/conda init bash
$HOME/miniconda3/bin/conda init zsh   # if you use zsh

# Reload shell
source ~/.bashrc   # or: exec zsh

# Verify
conda --version

## Option B: Install uv (fast, modern, replaces pip/venv/pipx)`

# Install uv (Rust-based, very fast)
curl -LsSf https://astral.sh/uv/install.sh | sh

# Reload shell
source ~/.bashrc   # or: exec zsh

# Verify
uv --version

## Option C: Install both (recommended — use conda for ML envs, uv for quick scripts)

# Do both Option A and Option B above
# Then verify both:
conda --version && uv --version

# 🐍 STEP 2: CREATE ENVIRONMENTS — ALL METHODS

## 🟢 METHOD 1: conda (best for PyTorch/CUDA, heavy ML stacks)


# --- Create named env (recommended) ---
conda create -n ds-research python=3.11 -y

# --- Activate ---
conda activate ds-research

# --- Install PyTorch with CUDA 12.1 (your GTX 1650) ---
conda install pytorch torchvision torchaudio pytorch-cuda=12.1 -c pytorch -c nvidia -y

# --- Install core DS stack ---
conda install numpy pandas matplotlib seaborn scikit-learn jupyterlab -y
conda install -c conda-forge transformers accelerate peft bitsandbytes -y
conda install -c conda-forge chromadb faiss-cpu langchain langgraph -y
conda install -c conda-forge mlflow wandb tensorboard -y

# --- Verify ---
python -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))"

# --- Deactivate ---
conda deactivate

conda env list
conda env remove -n ds-research

## 🟢 METHOD 2: uv + venv (fastest, best for reproducible lockfiles)


cd /home/lap-19/Desktop/DataScience-Reseach/DataScienceResearch/

# --- Create venv in project folder ---
uv venv .venv-ds-research --python 3.11

# --- Activate ---
source .venv-ds-research/bin/activate

# --- Install PyTorch with CUDA 12.1 (uv is 10-100x faster than pip) ---
uv pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# --- Install core DS stack (all at once, resolved fast) ---
uv pip install \
  numpy pandas matplotlib seaborn scikit-learn jupyterlab \
  transformers accelerate peft bitsandbytes \
  chromadb faiss-cpu langchain langgraph \
  mlflow wandb tensorboard \
  ipykernel

# --- Register kernel for Jupyter ---
uv pip install ipykernel
python -m ipykernel install --user --name=.venv-ds-research --display-name "Python (ds-research-uv)"

# --- Verify ---
python -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))"

# --- Deactivate ---
deactivate

# Generate lockfile
uv pip freeze > requirements-lock.txt

# Recreate from lockfile later
uv venv .venv-new --python 3.11
source .venv-new/bin/activate
uv pip install -r requirements-lock.txt

ls -la .venv-*/
rm -rf .venv-ds-research

## 🟢 METHOD 3: uv + conda (hybrid — conda for CUDA, uv for Python pkgs)

# Create base env with conda (gets CUDA toolkit, cudnn, etc.)
conda create -n ds-hybrid python=3.11 -y
conda activate ds-hybrid
conda install pytorch-cuda=12.1 -c pytorch -c nvidia -y  # CUDA runtime only

# Now use uv inside conda env for Python packages (much faster)
uv pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
uv pip install numpy pandas matplotlib seaborn scikit-learn jupyterlab
uv pip install transformers accelerate peft bitsandbytes
uv pip install chromadb faiss-cpu langchain langgraph mlflow wandb tensorboard

conda deactivate

## 🟢 METHOD 4: venv + pip (stdlib only, no extra tools)

cd /home/lap-19/Desktop/DataScience-Reseach/DataScienceResearch/

# --- Create ---
python3 -m venv .venv-ds-pip

# --- Activate ---
source .venv-ds-pip/bin/activate

# --- Upgrade pip ---
pip install --upgrade pip

# --- Install PyTorch CUDA 12.1 ---
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# --- Core stack ---
pip install numpy pandas matplotlib seaborn scikit-learn jupyterlab
pip install transformers accelerate peft bitsandbytes
pip install chromadb faiss-cpu langchain langgraph
pip install mlflow wandb tensorboard
pip install ipykernel

# --- Register kernel ---
python -m ipykernel install --user --name=.venv-ds-pip --display-name "Python (ds-pip)"

# --- Verify ---
python -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))"

deactivate

## 🟢 METHOD 5: Project-specific envs per Phase (recommended structure)

cd /home/lap-19/Desktop/DataScience-Reseach/DataScienceResearch/

# Phase 0-1: Math + Python (lightweight, no GPU needed)
uv venv .venv-phase01 --python 3.11
source .venv-phase01/bin/activate
uv pip install numpy pandas matplotlib jupyterlab sympy
deactivate

# Phase 3: Classical ML (CPU only, heavy on sklearn/xgb)
uv venv .venv-phase3 --python 3.11
source .venv-phase3/bin/activate
uv pip install scikit-learn xgboost lightgbm catboost optuna jupyterlab
deactivate

# Phase 4-7: Deep Learning + NLP + CV (needs CUDA)
uv venv .venv-phase47 --python 3.11
source .venv-phase47/bin/activate
uv pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
uv pip install transformers accelerate peft bitsandbytes
uv pip install timm albumentations opencv-python
uv pip install jupyterlab mlflow wandb
deactivate

# Phase 13-15: Local LLM + RAG + Agents (Ollama + lightweight)
uv venv .venv-phase1315 --python 3.11
source .venv-phase1315/bin/activate
uv pip install ollama chromadb faiss-cpu langchain langgraph
uv pip install streamlit gradio fastapi uvicorn
deactivate

# ⚡ QUICK REFERENCE CARD

---

## ⚡ QUICK REFERENCE CARD

| Task | conda | uv + venv | venv + pip |
|------|-------|-----------|------------|
| Create | `conda create -n name python=3.11` | `uv venv .venv-name --python 3.11` | `python3 -m venv .venv-name` |
| Activate | `conda activate name` | `source .venv-name/bin/activate` | `source .venv-name/bin/activate` |
| Install (fast) | `conda install pkg` | `uv pip install pkg` | `pip install pkg` |
| Install (CUDA PyTorch) | `conda install pytorch-cuda=12.1 -c pytorch -c nvidia` | `uv pip install torch --index-url https://download.pytorch.org/whl/cu121` | `pip install torch --index-url https://download.pytorch.org/whl/cu121` |
| Freeze/Export | `conda env export > env.yml` | `uv pip freeze > requirements.txt` | `pip freeze > requirements.txt` |
| Recreate from file | `conda env create -f env.yml` | `uv pip install -r requirements.txt` | `pip install -r requirements.txt` |
| List envs | `conda env list` | `ls .venv-*` | `ls .venv-*` |
| Remove env | `conda env remove -n name` | `rm -rf .venv-name` | `rm -rf .venv-name` |
| Register Jupyter kernel | `conda install ipykernel && python -m ipykernel install --user --name=name` | `uv pip install ipykernel && python -m ipykernel install --user --name=name` | `pip install ipykernel && python -m ipykernel install --user --name=name` |

---
